In [ ]:
import pandas as pd
import numpy as np
import os
from google.cloud import bigquery
from dotenv import load_dotenv
from scipy.stats import chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

# ==========================================
# 1. GOOGLE BIGQUERY CONNECTION
# ==========================================
load_dotenv('../.env')

# IMPORTANT: Ensure the exact name of your JSON file is placed here
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '../gcp_key.json' 

client = bigquery.Client()

print("Downloading data for the A/B Test from BigQuery...")

# Query your cloud table directly
query = """
    SELECT campaign_id, impressions, clicks, orders 
    FROM `mktng-performance-dashboard.data.marketing_kpis`
"""
df = client.query(query).to_dataframe()

print("Data ready! Preparing the competitors...\n")

# ==========================================
# 2. DATA PREPARATION
# ==========================================
# Group data by campaign to get the totals
df_grouped = df.groupby('campaign_id')[['impressions', 'clicks', 'orders']].sum().reset_index()

# Sort by impressions to select the top two largest campaigns
df_grouped = df_grouped.sort_values(by='impressions', ascending=False).head(2)

# Display the table to verify the selected data
display(df_grouped)

# Extract data for the two campaigns (Campaign A and Campaign B)
camp_a_id = df_grouped.iloc[0]['campaign_id']
camp_b_id = df_grouped.iloc[1]['campaign_id']

impressions_arr = [df_grouped.iloc[0]['impressions'], df_grouped.iloc[1]['impressions']]
clicks_arr = [df_grouped.iloc[0]['clicks'], df_grouped.iloc[1]['clicks']]
orders_arr = [df_grouped.iloc[0]['orders'], df_grouped.iloc[1]['orders']]

print(f"--- MATCHUP: Campaign {camp_a_id} vs Campaign {camp_b_id} ---\n")

Descargando datos para el A/B Test desde BigQuery...
¡Datos listos! Preparando a los competidores...



,campaign_id,impressions,clicks,orders
1,39889,1068337427,420003,1566
7,983498,137806768,509992,313


--- ENFRENTAMIENTO: Campaña 39889 vs Campaña 983498 ---

CTR Campaña A: 0.04% | CTR Campaña B: 0.37%
P-Valor del Z-Test: 0.0000
✅ Resultado: Con un 95% de confianza, la Campaña B es superior en generar Clics.

